In [326]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [327]:
import numpy as np
import matplotlib.pyplot as plt
import math
import time
from src.model import FiLMResNet2In, flatten_last
from src.normalizer import RunningMeanStd
from src.envpacker import packenv, packenv_batch
from src.utils import transition_ability_batched, update_v_history
import torch
from torch import nn
import torch.nn.functional as F

## 1. Configs

In [328]:
# Training configs
AGENTS = 50     # number of agents
LEARNING_RATE = 1e-3
TRAINING_STEPS = 30000
BATCH_SIZE = 10
DISPLAY_STEP = 1000 # For visualization
TRAIN_STEP_INTERVAL = 2 # Interval of steps between training episodes


# Bewley model parameters
theta = 1 # CRRA
beta = 0.975 # Discount factor
A = 1 # Technology parameter
alpha = 0.33 # Capital share of income
gamma = 2 # Inverse Frisch elasticity
########################################### (MiLF inputs)
r = 0.04 # Interest rate (Return to savings)
w = 1 # Wage rate (Return to labor)
delta = 0.06 # Depreciation rate of capital
TAX_PARAMS = {
    "tax_consumption": 0.065,          # Consumption tax (fixed)
    "tax_income": 0.2,                # Tax on labor income
    "income_tax_elasticity": 0.5,     # Elasticity of labor supply w.r.t. after-tax income
    "saving_tax_elasticity": 0.5,     # Elasticity of savings w.r.t. after-tax income
    "tax_saving": 0.1                 # Tax on interest income
}
###########################################
p = 2.2e-6
q = 0.99

# shock parameters
# log e' = rho_v * log e + sigma_v * epsilon, epsilon ~ N(0,1)
rho_v = 0.95 # persistence of ability shock
sigma_v = 0.2 # std of ability shock
v_bar = 1.5


In [329]:
# Bounds of shock 
v_min = math.exp(-2 * sigma_v  / math.sqrt(1-rho_v**2))
v_max = math.exp( 2 * sigma_v  / math.sqrt(1-rho_v**2))

## 2. Helper functions and classes

In [330]:
def mean_across_agents(x): # Since agent number is fix, thus we can use mean instead of sum
    return torch.mean(x, dim=1, keepdim=True)


def calculate_price(savings, ability, labor):
    savings_aggregate, labor_aggregate_effective = mean_across_agents(savings), mean_across_agents(labor * ability)
    wage = A * (1-alpha) * ((savings_aggregate/labor_aggregate_effective) ** alpha)
    ret = A * alpha * (savings_aggregate/labor_aggregate_effective ** alpha)
    return wage, ret

def taxfunc(ibt, abt, taxparams=TAX_PARAMS):
    it = ibt - (1 - taxparams["tax_income"]) * (ibt**(1-taxparams["income_tax_elasticity"])/(1-taxparams["income_tax_elasticity"])) # individual after tax income
    at = abt - (1-taxparams["tax_saving"]/1-taxparams["saving_tax_elasticity"]) * (abt**(1-taxparams["saving_tax_elasticity"])) # individual after tax saving
    return it, at

def calculate_moneydisposable(wage, ret, ability, labor, savings, delta, is_init=False):
    if is_init:
        ibt = wage * labor * ability   # individual before tax income
    else:
        ibt = wage * labor * ability + (1-delta+ret) * savings   # individual before tax income

    it, at = taxfunc(ibt = ibt, abt=savings)
    money_disposable = it + at

    return money_disposable, ibt


def output_transform(savings, money_disposable):

    # The a here is saving rate coming from the NN output
    consumption = money_disposable * (1 - savings)
    savings = money_disposable * savings    
    return consumption, savings


def laborfocloss(savings, labor, ibt, money_disposable, wage, ability, taxparams=TAX_PARAMS):

    loss_foc =  -labor ** (-gamma) + ((1-savings)*money_disposable/(1+taxparams["tax_consumption"])) * \
        (wage * ability) * (1 - taxparams["tax_income"]) * (ibt ** (-taxparams["income_tax_elasticity"]))

    return torch.abs(loss_foc)




In [331]:
state_dim = 2*AGENTS + 2 # two state variables for each agent + 2 individual variables
cond_dim = 5 # exogenous variables for all agents in all worlds(Batch)
model = FiLMResNet2In(state_dim=state_dim, cond_dim=cond_dim,
                        hidden_dim=128, num_res_blocks=3, output_dim=3, dropout=0.1)

In [332]:
def initial_state(required_batch_size, tax_params_dict=TAX_PARAMS):
    # 隨機產生初始資產與儲蓄
    moneydisposable = np.random.lognormal(0.1, 2.0, required_batch_size * AGENTS).reshape(required_batch_size, AGENTS)
    savings = np.random.lognormal(0.1, 2.0, required_batch_size * AGENTS).reshape(required_batch_size, AGENTS)

    # Initial productivity
    ability = np.random.lognormal(v_min, v_max, required_batch_size * AGENTS).reshape(required_batch_size, AGENTS)
    ability = ability / np.mean(ability, axis=1, keepdims=True)


    # superstar 標誌 (v1 對應一組, v2 對應一組)
    is_superstar_v1 = np.zeros((required_batch_size, AGENTS), dtype=bool)
    is_superstar_v2 = np.zeros((required_batch_size, AGENTS), dtype=bool)

    # 稅制參數轉為 tensor
    tax_params = torch.tensor(list(tax_params_dict.values()), dtype=torch.float32)
    tax_params = tax_params.repeat(required_batch_size, 1)

    # 轉為 tensor
    moneydisposable_t = torch.tensor(moneydisposable, dtype=torch.float32)
    savings_t = torch.tensor(savings, dtype=torch.float32)
    ability_t = torch.tensor(ability, dtype=torch.float32)
    is_superstar_vA = torch.tensor(is_superstar_v1, dtype=torch.bool)
    is_superstar_vB = torch.tensor(is_superstar_v2, dtype=torch.bool)
    tax_params_t = tax_params

    # 回傳字典
    return {
        "moneydisposable": {"value": moneydisposable_t, "shape": tuple(moneydisposable_t.shape)},
        "savings": {"value": savings_t, "shape": tuple(savings_t.shape)},
        "ability": {"value": ability_t, "shape": tuple(ability_t.shape)},
        "is_superstar_vA": {"value": is_superstar_vA, "shape": tuple(is_superstar_vA.shape)},
        "is_superstar_vB": {"value": is_superstar_vB, "shape": tuple(is_superstar_vB.shape)},
        "tax_params": {"value": tax_params_t, "shape": tuple(tax_params_t.shape)},
    }


In [333]:
def build_inputs(moneydisposable, v, tax_params):
    """
    回傳:
      features : (B, A, 2A + 2)          # 給模型輸入
      condi    : (B, A, Z)              # 稅制條件
      env_info : dict                   # 僅供環境轉移使用，不進模型
    """
    B, A = moneydisposable.shape

    # (B, Z) -> (B, A, Z)
    condi = tax_params.unsqueeze(1).expand(-1, A, -1)

    # (B, 2A) -> (B, A, 2A)
    sum_info = torch.cat([moneydisposable, v], dim=1)         # (B, 2A)
    sum_info_rep = sum_info.unsqueeze(1).expand(-1, A, -1)    # (B, A, 2A)

    # (B, A, 1) × 2
    money_self = moneydisposable.unsqueeze(-1)  # (B, A, 1)
    v_self     = v.unsqueeze(-1)                # (B, A, 1)

    # 給模型的 features
    features = torch.cat([sum_info_rep, money_self, v_self], dim=2)  # (B, A, 2A+2)

    return features, condi


In [334]:
def run_network_given_state(state, model, brach=None):
    
    # 把整個state agent 需要作動的部分取出來

    if not brach:
        agent_state = build_inputs(
            moneydisposable=state["moneydisposable"]["value"],
            v=state["ability"]["value"],
            tax_params=state["tax_params"]["value"]
        )

    else:
        # print(state["moneydisposable"])
        agent_state = build_inputs(
            moneydisposable=state["moneydisposable"],
            v=state[f"ability_{brach}"],
            tax_params=state["tax_params"]
        )

    acts = [torch.sigmoid, lambda x: F.softplus(x) + 1e-6, torch.sigmoid]
    out = model(agent_state[0], agent_state[1])
    savings_t1, mu_t0, labor_t0 = [acts[i](out[..., i]) for i in range(out.shape[-1])]
    savings_t1, mu_t0, labor_t0 = savings_t1.squeeze(-1), mu_t0.squeeze(-1), labor_t0.squeeze(-1)

    return {
        "savings_t1": savings_t1,
        "mu_t0": mu_t0,
        "labor_t0": labor_t0
    }



In [335]:
def part_transition_transform(state, model_out, branch=None):
    
    # Current wage and return
    wage, ret = calculate_price(
        savings=state["savings"]["value"] if not branch else state["savings"],
        ability=state["ability"]["value"] if not branch else state[f"ability_{branch}"],
        labor=model_out["labor_t0"],
    )

    money_disposable, ibt = calculate_moneydisposable(wage=wage, ret=ret, 
                                                ability=state["ability"]["value"] if not branch else state[f"ability_{branch}"],
                                                labor=model_out["labor_t0"], savings=state["savings"]["value"] if not branch else state["savings"],
                                                delta=delta)
    
    consumption, savings = output_transform(savings=model_out["savings_t1"], money_disposable=money_disposable)

    return {
        "moneydisposable": money_disposable,
        "savings": savings, # savings for next period
        "consumption": consumption, # consumption for current period
        "wage": wage,
        "ret": ret,
        "ibt": ibt,
        "tax_params": state["tax_params"]["value"] if not branch else state["tax_params"]
    }

In [336]:
# 在這裡加入shock
# def update_state_dict(part_transition_output, state_dicta):
#     state_dict["moneydisposable"] = part_transition_output["moneydisposable"]
#     state_dict["savings"] = part_transition_output["savings"]
#     state_dict[]

In [337]:
def run_model(state_dict_train, model, v_history_A, v_history_B):

    state_dict_tmp = {}
    # t=0
    model_out_0 = run_network_given_state(state_dict_train, model=model)

    # moneydisposable, savings, consumption, wage, ret, ibt
    part_transition_output = part_transition_transform(state=state_dict_train, model_out=model_out_0)
    keys_to_keep = ["moneydisposable", "savings", "tax_params"]
    state_dict_tmp = {k: part_transition_output[k] for k in keys_to_keep if k in part_transition_output}

    v_next_A, is_superstar_next_A = transition_ability_batched(
        ability=state_dict_train["ability"]["value"],
        is_superstar_prev=state_dict_train["is_superstar_vA"]["value"],
        v_history=v_history_A,
        rho_v=rho_v, sigma_v=sigma_v,
        p=p, q=q, v_bar=v_bar,
        v_min=v_min, v_max=v_max
    )

    v_next_B, is_superstar_next_B = transition_ability_batched(
        ability=state_dict_train["ability"]["value"],
        is_superstar_prev=state_dict_train["is_superstar_vB"]["value"],
        v_history=v_history_A,
        rho_v=rho_v, sigma_v=sigma_v,
        p=p, q=q, v_bar=v_bar,
        v_min=v_min, v_max=v_max
    )

    v_history_A = update_v_history(
        v_history=v_history_A,
        v_next=v_next_A
    )
    v_history_B = update_v_history(
        v_history=v_history_B,
        v_next=v_next_B
    )

    state_dict_tmp["ability_A"] = v_next_A
    state_dict_tmp["ability_B"] = v_next_B


    # t=1
    model_out_1A = run_network_given_state(state_dict_tmp, model=model, brach="A")
    model_out_1B = run_network_given_state(state_dict_tmp, model=model, brach="B")

    part_transition_output_A = part_transition_transform(state_dict_tmp, model_out_1A, branch="A")
    part_transition_output_B = part_transition_transform(state_dict_tmp, model_out_1B, branch="B")


    
    state_dict_train["is_superstar_vA"] = is_superstar_next_A 
    state_dict_train["is_superstar_vB"] = is_superstar_next_B

    return state_dict_train, v_history_A, v_history_B

In [338]:
# run_model(state_dict_train, model, v_history_A=v_history_A, v_history_B=v_history_B)

In [339]:
state_dict_train = initial_state(required_batch_size=256)
v_history_A = None
v_history_B = None

for i in range(5):
    if i % TRAIN_STEP_INTERVAL == 0:
        print(i)
        state_dict_train, v_history_A, v_history_B = run_model(state_dict_train = state_dict_train, 
                                                               model = model, 
                                                               v_history_A=v_history_A, 
                                                               v_history_B=v_history_B)
        print('='*50)


0
2


IndexError: too many indices for tensor of dimension 2